In [9]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""Basal DEG logfoldchange heatmap (genes x datasets)

You said your DEG files look like:  R-CG_ranked_genes.csv
Columns:
  - gene           : gene name
  - logfoldchange  : logFC value
  - group          : cell group name (e.g., Basal, LumSEC, ...)

This script:
  1) reads each dataset's "*_ranked_genes.csv"
  2) filters rows where group == GROUP_NAME (default: "Basal")
  3) extracts logfoldchange for a specified GENES list
  4) plots one heatmap: rows=genes, cols=datasets

Edit the GLOBAL VARIABLES section and run:
  python Basal_DEG_logFC_heatmap.py

Outputs:
  - HEATMAP_OUT (pdf/png)
  - MATRIX_OUT  (tsv)
"""

from __future__ import annotations

from pathlib import Path
from typing import List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

# =========================
# GLOBAL VARIABLES (EDIT ME)
# =========================

# 6 datasets (column order in the heatmap)
DATASETS = ["M-MG", "R-MG", "S-MG", "S-AG", "R-AG", "R-CG"]

# Which cell group to extract from column "group"
GROUP_NAME = "Basal"
GENES = ['Lcn2','Trp63','Acta2','Myh11','Tpm2','Mylk','Myl9','Actg2']
# Outputs
HEATMAP_OUT = Path(r"D:/111/Basal_logfoldchange_heatmap.pdf")
MATRIX_OUT = Path(r"D:/111/Basal_logfoldchange_heatmap.matrix.tsv")

# Plot options
TITLE = "Basal DEG logfoldchange (genes x datasets)"
ANNOTATE_VALUES = False  # True if your matrix is small
CENTER = 0.0

# If None, the script auto-sets symmetric vmin/vmax around 0 using 98th percentile
VMIN: Optional[float] = -5
VMAX: Optional[float] = 5

# =========================
# IMPLEMENTATION
# =========================

REQUIRED_COLS = {"gene", "logfoldchange", "group"}

def build_matrix(genes: List[str]) -> pd.DataFrame:
    mat = pd.DataFrame(index=genes, columns=DATASETS, dtype=float)

    for ds in DATASETS:
        fp = f"D:/111/{ds}_ranked_genes.csv"
        df = pd.read_csv(fp)

        missing = REQUIRED_COLS - set(df.columns)
        if missing:
            raise KeyError(
                f"Missing columns {missing} in {fp}. Found columns: {list(df.columns)}"
            )

        # Filter to target group (e.g., Basal)
        sub = df.loc[df["group"].astype(str) == GROUP_NAME, ["gene", "logfoldchange"]].copy()
        sub["gene"] = sub["gene"].astype(str)

        # If duplicates exist for the same gene within the group, keep max-abs logFC
        if sub["gene"].duplicated().any():
            sub["__abs__"] = sub["logfoldchange"].abs()
            sub = sub.sort_values("__abs__", ascending=False).drop_duplicates("gene", keep="first")
            sub = sub.drop(columns=["__abs__"])

        s = sub.set_index("gene")["logfoldchange"]

        mat[ds] = [float(s.get(g, np.nan)) for g in genes]

    return mat


def plot_heatmap(mat: pd.DataFrame) -> None:
    values = mat.to_numpy(dtype=float)
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        raise ValueError("All values are NaN; nothing to plot.")

    vmin = VMIN
    vmax = VMAX
    if vmin is None or vmax is None:
        q = np.nanpercentile(np.abs(finite), 98)
        if q == 0:
            q = float(np.nanmax(np.abs(finite)))
        if q == 0:
            q = 1.0
        if vmin is None:
            vmin = -q
        if vmax is None:
            vmax = +q

    norm = TwoSlopeNorm(vmin=vmin, vcenter=CENTER, vmax=vmax)

    # Auto figure size
    w = max(6.0, 1.1 * len(mat.columns))
    h = max(6.0, 0.35 * len(mat.index))

    fig, ax = plt.subplots(figsize=(w, h))
    im = ax.imshow(values, aspect="auto", interpolation="nearest", norm=norm,cmap="RdBu_r")

    ax.set_title(TITLE, pad=12)
    ax.set_xticks(np.arange(len(mat.columns)))
    ax.set_xticklabels(mat.columns, rotation=45, ha="right")
    ax.set_yticks(np.arange(len(mat.index)))
    ax.set_yticklabels(mat.index)

    # light grid
    ax.set_xticks(np.arange(-.5, len(mat.columns), 1), minor=True)
    ax.set_yticks(np.arange(-.5, len(mat.index), 1), minor=True)
    ax.grid(which="minor", linestyle="-", linewidth=0.5)
    ax.tick_params(which="minor", bottom=False, left=False)

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("logfoldchange", rotation=90)

    if ANNOTATE_VALUES:
        for i in range(values.shape[0]):
            for j in range(values.shape[1]):
                v = values[i, j]
                if np.isfinite(v):
                    ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8)

    fig.tight_layout()
    HEATMAP_OUT.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(HEATMAP_OUT, dpi=300)
    plt.close(fig)


def main():
    if not DEG_DIR.exists():
        raise FileNotFoundError(f"DEG_DIR does not exist: {DEG_DIR}")

    genes = GENES
    mat = build_matrix(genes)

    MATRIX_OUT.parent.mkdir(parents=True, exist_ok=True)
    mat.to_csv(MATRIX_OUT, sep="	", index=True)

    plot_heatmap(mat)

    print(f"[OK] group: {GROUP_NAME}")
    print(f"[OK] saved matrix:  {MATRIX_OUT}")
    print(f"[OK] saved heatmap: {HEATMAP_OUT}")


if __name__ == "__main__":
    main()


[OK] group: Basal
[OK] saved matrix:  D:\111\Basal_logfoldchange_heatmap.matrix.tsv
[OK] saved heatmap: D:\111\Basal_logfoldchange_heatmap.pdf
